###Uploading data to Databricks volumes through UI

Data uploaded to below path: /Volumes/workspace/training/raw_files
- I am going to load data from raw files to bronze database
- I have created a directory called _bronze_ where all the 6 files placed and tables created accordingly.

In [0]:
%sql
--- I am going to create 3 databases bronze, silver and gold
create database IF NOT EXISTS bronze;
create database IF NOT EXISTS silver;
create database IF NOT EXISTS gold;
show databases;

databaseName
bronze
default
gold
information_schema
silver
training


In [0]:
dbutils.fs.ls("/Volumes/workspace/training/raw_files")

[FileInfo(path='dbfs:/Volumes/workspace/training/raw_files/media_customer_reviews.parquet', name='media_customer_reviews.parquet', size=46004, modificationTime=1747730000000),
 FileInfo(path='dbfs:/Volumes/workspace/training/raw_files/media_gold_reviews_chunked.parquet', name='media_gold_reviews_chunked.parquet', size=23557, modificationTime=1747730000000),
 FileInfo(path='dbfs:/Volumes/workspace/training/raw_files/sales_customers.parquet', name='sales_customers.parquet', size=28493, modificationTime=1747730000000),
 FileInfo(path='dbfs:/Volumes/workspace/training/raw_files/sales_franchises.parquet', name='sales_franchises.parquet', size=5905, modificationTime=1747730000000),
 FileInfo(path='dbfs:/Volumes/workspace/training/raw_files/sales_suppliers.parquet', name='sales_suppliers.parquet', size=4591, modificationTime=1747730000000),
 FileInfo(path='dbfs:/Volumes/workspace/training/raw_files/sales_transactions.parquet', name='sales_transactions.parquet', size=86578, modificationTime=17

In [0]:
## This function takes a path to a parquet file and returns a Spark DataFrame
def parquetToDF(path):
  df = spark.read.parquet(path)
  return df

In [0]:
%sql
show tables in workspace.bronze;

database,tableName,isTemporary
bronze,media_customer_reviews,false
bronze,media_gold_reviews_chunked,false
bronze,sales_customers,false
bronze,sales_franchises,false
bronze,sales_suppliers,false
bronze,sales_transactions,false
,_sqldf,true


In [0]:
%python
## this code will extract file names and tables from the source file
## By using same source file, creating delta tables

from pathlib import Path

# Path to your volume
volume_path = "/Volumes/workspace/training/raw_files/"

# List all files in the volume
files = dbutils.fs.ls(volume_path)
total_files = len(files)
print(f"Total number of files: {total_files}")

# Parquet file paths
parquet_paths = [f.path for f in files]

# Extract file names without the .parquet extension
for path in parquet_paths:
    file_path = Path(path)
    table_name = file_path.stem
    df = parquetToDF(path)
    df.write.format("delta").mode("overwrite").saveAsTable(f"workspace.bronze.{table_name}")
    
    print(f"{table_name}")

Total number of files: 6
media_customer_reviews
media_gold_reviews_chunked
sales_customers
sales_franchises
sales_suppliers
sales_transactions


In [0]:
%sql
show tables in workspace.bronze;

database,tableName,isTemporary
bronze,media_customer_reviews,false
bronze,media_gold_reviews_chunked,false
bronze,sales_customers,false
bronze,sales_franchises,false
bronze,sales_suppliers,false
bronze,sales_transactions,false
,_sqldf,true


Creating Silver tables from bronze tables

In [0]:
%python

for path in parquet_paths:
    file_path = Path(path)
    table_name = file_path.stem
    bronze_table = f"workspace.bronze.{table_name}"
    silver_table = f"workspace.silver.{table_name}_silver"

    print(f"table_name: {table_name}")
    print(f"bronze_table: {bronze_table}")
    print(f"silver_table: {silver_table}")
    spark.sql(f"drop table if exists {silver_table}")
    spark.sql(f"create or replace table {silver_table} as select * from {bronze_table}")
    print(f"Created table {silver_table}")


table_name: media_customer_reviews
bronze_table: workspace.bronze.media_customer_reviews
silver_table: workspace.silver.media_customer_reviews_silver
Created table workspace.silver.media_customer_reviews_silver
table_name: media_gold_reviews_chunked
bronze_table: workspace.bronze.media_gold_reviews_chunked
silver_table: workspace.silver.media_gold_reviews_chunked_silver
Created table workspace.silver.media_gold_reviews_chunked_silver
table_name: sales_customers
bronze_table: workspace.bronze.sales_customers
silver_table: workspace.silver.sales_customers_silver
Created table workspace.silver.sales_customers_silver
table_name: sales_franchises
bronze_table: workspace.bronze.sales_franchises
silver_table: workspace.silver.sales_franchises_silver
Created table workspace.silver.sales_franchises_silver
table_name: sales_suppliers
bronze_table: workspace.bronze.sales_suppliers
silver_table: workspace.silver.sales_suppliers_silver
Created table workspace.silver.sales_suppliers_silver
table_nam

Adding SCD type-1 changes to every table


In [0]:
from delta.tables import DeltaTable

# Get a list of all tables in a catalog and database
def MergeInto_silver(bronze_table_name, key_col):
    
    print(f"bronze_table_name: {bronze_table_name}")
    bronze_df = spark.sql(f"select * from {bronze_table_name} ")
    
    silver_table = f"{bronze_table_name}_silver".replace("bronze", "silver")
    print(f"silver_table: {silver_table}")
    silver_df = spark.sql(f"select * from {silver_table} \n")

    merge_condition = f"target.{key_col} = source.{key_col}"

    delta_table = DeltaTable.forName(spark, silver_table)
    
    delta_table.alias("target") \
        .merge(bronze_df.alias("source"),
               merge_condition) \
                   .whenMatchedUpdateAll(condition=f"""
            target.{key_col} != source.{key_col}""") \
                       .whenNotMatchedInsertAll() \
                           .execute()
    

MergeInto_silver("workspace.bronze.media_gold_reviews_chunked", "franchiseID")
MergeInto_silver("workspace.bronze.media_customer_reviews", "franchiseID")
MergeInto_silver("workspace.bronze.sales_customers", "customerID")
MergeInto_silver("workspace.bronze.sales_franchises", "franchiseID")
MergeInto_silver("workspace.bronze.sales_suppliers", "supplierID")
MergeInto_silver("workspace.bronze.sales_transactions", "franchiseID")

#########################################################################
### Using SQL Queries for Merge Operation:

#MERGE INTO workspace.silver.sales_customers_silver AS target 
#USING workspace.bronze.sales_customers AS source 
#ON target.customerID = source.customerID 
#WHEN MATCHED THEN UPDATE SET customerID = source.customerID, first_name = source.first_name, last_name = #source.last_name, email_address = source.email_address, phone_number = source.phone_number, address = #source.address, city = source.city, state = source.state, country = source.country, continent = source.#continent, postal_zip_code = source.postal_zip_code, gender = source.gender
#WHEN NOT MATCHED THEN INSERT *;
#########################################################################


bronze_table_name: workspace.bronze.media_gold_reviews_chunked
silver_table: workspace.silver.media_gold_reviews_chunked_silver
bronze_table_name: workspace.bronze.media_customer_reviews
silver_table: workspace.silver.media_customer_reviews_silver
bronze_table_name: workspace.bronze.sales_customers
silver_table: workspace.silver.sales_customers_silver
bronze_table_name: workspace.bronze.sales_franchises
silver_table: workspace.silver.sales_franchises_silver
bronze_table_name: workspace.bronze.sales_suppliers
silver_table: workspace.silver.sales_suppliers_silver
bronze_table_name: workspace.bronze.sales_transactions
silver_table: workspace.silver.sales_transactions_silver


### Gold layer tables creation
1.	Get the most sold products to identify the top-selling items.

In [0]:
from pyspark.sql.functions import col,sum

### using spark sql
spark.sql("select product,sum(quantity) from workspace.silver.sales_transactions_silver group by product").display()

### using pyspark
trans_silver_df = spark.sql("select * from workspace.silver.sales_transactions_silver")
trans_silver_df = trans_silver_df.groupBy(col("product")).agg(sum(col("quantity")).alias("total_sold_products"))

### writing data into gold layer
trans_silver_df.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.total_sol_products")



product,sum(quantity)
Pearly Pies,3595
Outback Oatmeal,3733
Austin Almond Biscotti,3716
Orchard Oasis,3586
Tokyo Tidbits,3662
Golden Gate Ginger,3865


In [0]:
%sql
select * from gold.total_sol_products;

product,total_sold_products
Pearly Pies,3595
Outback Oatmeal,3733
Austin Almond Biscotti,3716
Orchard Oasis,3586
Tokyo Tidbits,3662
Golden Gate Ginger,3865


2.	Find which suppliers provide ingredients to the most franchises.

In [0]:
from pyspark.sql.functions import col,count

#reading data from franchise and supplier tables
franchise_sil_df = spark.read.table("workspace.silver.sales_franchises_silver")
suppliers_sil_df = spark.read.table("workspace.silver.sales_suppliers_silver")

##joining two tables
franchise_supply_df = franchise_sil_df.join(suppliers_sil_df, ["supplierID"], "inner")

## counting the franchiseIDs and ingredients based on supplierID
join_df = franchise_supply_df.groupby(col("supplierID")) \
                 .agg(count(col("franchiseID")).alias("franchise_count"), 
                      count(col("ingredient")).alias("ingredient_count")) \
                 .orderBy(col("supplierID").asc(), col("ingredient_count").desc())

## writing data into gold layer
join_df.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.supplier_franchise_ingredients")


In [0]:
%sql
select count(*) from gold.supplier_franchise_ingredients;

count(*)
27


3.	Get total sales per month.

In [0]:
from pyspark.sql.functions import sum, col, year, month

# Load sales_transactions data
sale_trans_df = spark.read.table("workspace.silver.sales_transactions_silver")
sale_trans_df = sale_trans_df.withColumn("year", year(col("dateTime"))) \
                            .withColumn("month", month(col("dateTime")))


sale_trans_df = sale_trans_df.groupBy("year", "month").agg(sum(col("totalPrice")).alias("sales_by_month"))
sale_trans_df.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.sales_by_month")


In [0]:
%sql
select * from workspace.gold.sales_by_month;

year,month,sales_by_month
2024,5,66471
